# CardioIA - Fase 2 - Parte 1: Extracao de Sintomas e Diagnostico

**FIAP 2026** | Grupo 63

Este notebook implementa um sistema basico de apoio ao diagnostico cardiologico usando NLP simples.

### Objetivo:
- Ler 10 frases de sintomas de pacientes
- Identificar sintomas com base em um mapa de conhecimento
- Sugerir diagnostico associado ao sintoma encontrado

In [ ]:
# Instalacao de dependencias (se necessario)
# !pip install pandas

## 1. Importacao de Bibliotecas

In [ ]:
import pandas as pd
import re

print('Bibliotecas importadas com sucesso!')

## 2. Carregamento do Arquivo de Sintomas (.txt)

In [ ]:
# Leitura do arquivo de sintomas
with open('sintomas.txt', 'r', encoding='utf-8') as f:
    frases = f.readlines()

# Remover espacos e quebras de linha
frases = [frase.strip() for frase in frases if frase.strip()]

print(f'Total de frases carregadas: {len(frases)}')
for i, frase in enumerate(frases, 1):
    print(f'Paciente {i}: {frase}')

## 3. Carregamento do Mapa de Conhecimento (.csv)

In [ ]:
# Carregamento do mapa de conhecimento sintoma -> doenca
mapa_df = pd.read_csv('mapa_conhecimento.csv')
print(f'Total de regras no mapa: {len(mapa_df)}')
print(mapa_df.head(10))

## 4. Funcao de Identificacao de Sintomas e Sugestao de Diagnostico

In [ ]:
def normalizar_texto(texto):
    '''Normaliza o texto removendo acentos e convertendo para minusculas'''
    texto = texto.lower()
    substituicoes = {
        'a': ['a', 'a', 'a', 'a'],
        'e': ['e', 'e'],
        'i': ['i', 'i'],
        'o': ['o', 'o', 'o'],
        'u': ['u', 'u'],
        'c': ['c']
    }
    return texto

def identificar_diagnostico(frase, mapa_df):
    '''Identifica o diagnostico com base nos sintomas presentes na frase'''
    frase_lower = frase.lower()
    diagnosticos_encontrados = []
    
    for _, row in mapa_df.iterrows():
        # Verifica se algum dos sintomas esta presente na frase
        sintomas = [str(row['Sintoma1']).lower(), 
                    str(row['Sintoma2']).lower(), 
                    str(row['Sintoma3']).lower()]
        
        for sintoma in sintomas:
            # Verifica palavras-chave do sintoma
            palavras_chave = sintoma.split()[:2]  # Primeiras 2 palavras
            if all(p in frase_lower for p in palavras_chave):
                diagnostico = row['Doenca_Associada']
                if diagnostico not in diagnosticos_encontrados:
                    diagnosticos_encontrados.append(diagnostico)
                break
    
    if diagnosticos_encontrados:
        return diagnosticos_encontrados[0]
    return 'Nao identificado - encaminhar para avaliacao medica'

print('Funcao de diagnostico criada com sucesso!')

## 5. Aplicacao e Resultados

In [ ]:
# Aplicando o diagnostico para cada paciente
print('=' * 70)
print('SISTEMA CARDIOIA - SUGESTAO DE DIAGNOSTICO')
print('=' * 70)

resultados = []
for i, frase in enumerate(frases, 1):
    diagnostico = identificar_diagnostico(frase, mapa_df)
    resultados.append({'Paciente': i, 'Descricao': frase, 'Diagnostico Sugerido': diagnostico})
    print(f'\nPaciente {i}:')
    print(f'  Descricao: {frase[:80]}...')
    print(f'  Diagnostico: {diagnostico}')

print('\n' + '=' * 70)

# Criar DataFrame com resultados
resultados_df = pd.DataFrame(resultados)
print('\nResumo dos diagnosticos:')
print(resultados_df[['Paciente', 'Diagnostico Sugerido']])

## 6. Conclusao

Este sistema demonstra um classificador basico de sintomas cardiologicos que:
- Le descricoes textuais de pacientes
- Identifica palavras-chave associadas a condicoes cardiacas
- Sugere um diagnostico baseado no mapa de conhecimento

**Limitacoes:** Este e um sistema educativo simplificado. Nao substitui avaliacao medica profissional.

In [ ]:
# Salvar resultados em CSV
resultados_df.to_csv('resultados_diagnostico.csv', index=False)
print('Resultados salvos em resultados_diagnostico.csv')